In [ ]:
#!/usr/bin/env python3
# ======================================================================
#  CryoSat-2 TEMPO FTP downloader for POCA and SWATH point products.
#  Downloads Antarctica POCA and SWATH point products from 2010 through 2026.
#  CryoSat-2 data are available starting in April 2010; later months run
#  through December 2026. Existing FTP, retry, and parallel download
#  settings are retained.
# ======================================================================

import os
import time
import ftplib
import threading
from dataclasses import dataclass
from getpass import getpass
from typing import Tuple, Optional, List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

HOST       = "science-pds.cryosat.esa.int"
DEST_ROOT  = "/Volumes/icefall/marquardt/data/CS2/cs2_tempo_LI"

# ----------------------------
# PRODUCTS / EXTENSIONS
# ----------------------------
EXTS_BY_PRODUCT = {
    "TEMPO_POCA_LI":     (".nc",),
    "TEMPO_SWATH_POINT": (".nc",),
}

START_YEAR = 2010
END_YEAR   = 2026

# No phase-specific cutoffs: each enabled phase uses the full configured
# period, subject to CryoSat-2 data availability.
PHASE_CUTOFFS: Dict[str, Tuple[int, int]] = {}

# retry parameters
LIST_TIMEOUT = 50
RETRIES = 6
BACKOFF = [1, 2, 4, 8, 15, 30]
BLOCK = 1 << 20  # 1MB

# parallelism (reduced to lower pressure)
MAX_WORKERS = 4  # simultaneous file downloads

# force plain FTP (no TLS) to reduce server resets/throttling
FORCE_PLAIN_FTP = True

# ----------------------------
# PHASES (run in order)
# ----------------------------
PHASES: List[Dict] = [
    {
        "label":   "ANTARCTICA_POCA",
        "product": "TEMPO_POCA_LI",
        "leaves":  ["ANTARC"],
        "enabled": True,
    },
    {
        "label":   "ANTARCTICA_SWATH",
        "product": "TEMPO_SWATH_POINT",
        "leaves":  ["ANTARCTIC"],
        "enabled": True,
    },
]


def ts():
    import datetime as _dt
    return _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def cryosat_month_available(y: int, m: int) -> bool:
    if y < 2010:
        return False
    if y == 2010 and m < 4:
        return False
    return True


def is_before(y: int, m: int, y0: int, m0: int) -> bool:
    return (y < y0) or (y == y0 and m < m0)


def ftp_connect(user: str, pwd: str):
    """
    Connect using plain FTP (passive).
    Kept as a single function so you can re-enable FTPS later if desired.
    """
    # Forced plain FTP per requested change
    ftp = ftplib.FTP()
    ftp.connect(HOST, 21, timeout=LIST_TIMEOUT)
    ftp.login(user, pwd)
    ftp.set_pasv(True)
    return ftp, "FTP"


def nlst_safe(ftp, path: str) -> List[str]:
    """
    List names under path using existing connection.
    Fast-fail on 550; retry otherwise.
    """
    for i in range(RETRIES):
        try:
            entries: List[str] = []
            ftp.cwd(path)
            ftp.retrlines("NLST", entries.append)
            return [e.strip() for e in entries if e.strip()]
        except ftplib.error_perm as e:
            # 550 -> missing directory; treat as empty
            if "550" in str(e):
                return []
            return []
        except Exception:
            # Try to return to root and retry
            try:
                ftp.cwd("/")
            except Exception:
                pass
            time.sleep(BACKOFF[min(i, len(BACKOFF) - 1)])
    return []


@dataclass(frozen=True)
class Job:
    remote_dir: str
    name: str
    local_path: str
    year: int
    month: int
    month_idx: int
    month_total: int


class Worker:
    """
    Per-thread FTP session reused across many downloads.
    Uses plain FTP (passive) to reduce connection resets.
    """
    def __init__(self, user: str, pwd: str, worker_id: int):
        self.user = user
        self.pwd = pwd
        self.worker_id = worker_id
        self.ftp = None
        self.mode = None
        self.curdir = None

    def _close(self):
        if self.ftp is not None:
            try:
                self.ftp.quit()
            except Exception:
                try:
                    self.ftp.close()
                except Exception:
                    pass
        self.ftp = None
        self.mode = None
        self.curdir = None

    def _connect(self):
        self._close()
        ftp, mode = ftp_connect(self.user, self.pwd)
        self.ftp = ftp
        self.mode = mode
        self.curdir = None

    def ensure_connected(self):
        if self.ftp is None:
            self._connect()

    def ensure_cwd(self, remote_dir: str):
        self.ensure_connected()
        if self.curdir == remote_dir:
            return

        for i in range(RETRIES):
            try:
                self.ftp.cwd(remote_dir)
                self.curdir = remote_dir
                return
            except ftplib.error_perm as e:
                if "550" in str(e):
                    raise
                raise
            except Exception:
                time.sleep(BACKOFF[min(i, len(BACKOFF) - 1)])
                self._connect()

        raise RuntimeError(f"Failed to cwd to {remote_dir} after retries")

    def download(self, job: Job) -> str:
        """Return 'ok' | 'exists' | 'fail'."""
        os.makedirs(os.path.dirname(job.local_path), exist_ok=True)

        if os.path.exists(job.local_path):
            return "exists"

        tmp = job.local_path + ".part"
        resume_pos = os.path.getsize(tmp) if os.path.exists(tmp) else 0

        last_err = None
        for i in range(RETRIES):
            try:
                self.ensure_cwd(job.remote_dir)

                mode_file = "ab" if resume_pos > 0 else "wb"
                with open(tmp, mode_file) as f:
                    try:
                        self.ftp.retrbinary(
                            f"RETR {job.name}",
                            f.write,
                            blocksize=BLOCK,
                            rest=resume_pos if resume_pos > 0 else None,
                        )
                    except Exception:
                        # If resume failed mid-stream, drop partial and retry from 0
                        if resume_pos > 0:
                            try:
                                f.seek(0)
                                f.truncate(0)
                            except Exception:
                                pass
                            resume_pos = 0
                            self._connect()
                            raise
                        raise

                os.replace(tmp, job.local_path)
                return "ok"

            except ftplib.error_perm as e:
                if "550" in str(e):
                    return "fail"
                last_err = e
                return "fail"

            except Exception as e:
                last_err = e
                self._connect()
                time.sleep(BACKOFF[min(i, len(BACKOFF) - 1)])

        print(f"[{ts()}] [WARN] giving up on {job.remote_dir}/{job.name}: {last_err}")
        return "fail"


_tls = threading.local()

def _get_worker(user: str, pwd: str, worker_id_hint: int) -> Worker:
    w = getattr(_tls, "worker", None)
    if w is None:
        w = Worker(user, pwd, worker_id_hint)
        _tls.worker = w
    return w


def _download_task(user: str, pwd: str, job: Job, worker_id_hint: int) -> str:
    w = _get_worker(user, pwd, worker_id_hint)
    return w.download(job)


def cutoff_for_phase(phase_label: str) -> Optional[Tuple[int, int]]:
    """Return (year, month) cutoff for this specific phase label, or None."""
    return PHASE_CUTOFFS.get(phase_label)


def build_jobs_for_phase(user: str, pwd: str, phase: Dict) -> List[Job]:
    """
    CHANGED: Reuse one FTP connection for ALL listings in this phase.
    This avoids opening thousands of short-lived sessions (login storms).
    """
    label   = phase["label"]
    product = phase["product"]
    leaves  = phase["leaves"]
    exts    = EXTS_BY_PRODUCT.get(product, (".nc",))
    cf      = cutoff_for_phase(label)

    print(f"\n== PHASE: {label} ==")
    print(f"[INFO] Product: {product}")
    print(f"[INFO] Leaves: {leaves}")
    if cf:
        print(f"[INFO] {label}: starting from {cf[0]}-{cf[1]:02d} (earlier months skipped)")
    else:
        print(f"[INFO] {label}: no cutoff (processing full range)")
    print(f"[INFO] Listing mode: {'FTP (plain, forced)' if FORCE_PLAIN_FTP else 'FTP/FTPS'}")

    jobs: List[Job] = []

    ftp = None
    try:
        ftp, _mode = ftp_connect(user, pwd)

        for y in range(START_YEAR, END_YEAR + 1):
            for m in range(1, 13):
                if not cryosat_month_available(y, m):
                    continue
                if cf and is_before(y, m, cf[0], cf[1]):
                    continue

                found_any = False
                for leaf in leaves:
                    remote_dir = f"/{product}/{y}/{m:02d}/{leaf}"
                    print(f"[{ts()}] Listing {product} {y}-{m:02d} leaf={leaf} …")
                    names = nlst_safe(ftp, remote_dir)
                    files = [n for n in names if any(n.endswith(ext) for ext in exts)]

                    if files:
                        found_any = True
                        month_total = len(files)
                        print(f"  -> {month_total} files found under {remote_dir}/")

                        for month_idx, name in enumerate(files, start=1):
                            rel = os.path.join(product, f"{y}", f"{m:02d}", leaf, name)
                            local_path = os.path.join(DEST_ROOT, rel)
                            jobs.append(
                                Job(
                                    remote_dir=remote_dir,
                                    name=name,
                                    local_path=local_path,
                                    year=y,
                                    month=m,
                                    month_idx=month_idx,
                                    month_total=month_total,
                                )
                            )
                        break
                    else:
                        print(f"  -> no files at {remote_dir}/ (trying next leaf if any)")

                if not found_any:
                    print(f"  ..no leaf with files for {product} {y}-{m:02d}")

    finally:
        if ftp is not None:
            try:
                ftp.quit()
            except Exception:
                try:
                    ftp.close()
                except Exception:
                    pass

    print(f"[{ts()}] Phase queued jobs: {len(jobs)}")
    return jobs


def run_jobs(user: str, pwd: str, jobs: List[Job]) -> Tuple[int, int, int]:
    if not jobs:
        return (0, 0, 0)

    total_ok = 0
    total_skip = 0
    total_fail = 0

    print_lock = threading.Lock()

    def _submit_id(idx: int) -> int:
        return (idx % MAX_WORKERS) + 1

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {}
        for idx, job in enumerate(jobs, 1):
            worker_hint = _submit_id(idx)
            fut = ex.submit(_download_task, user, pwd, job, worker_hint)
            futures[fut] = job

        for fut in as_completed(futures):
            job = futures[fut]
            try:
                status = fut.result()
            except Exception as e:
                status = "fail"
                with print_lock:
                    print(f"[{ts()}] [ERR] {job.remote_dir}/{job.name}: {e}")

            if status == "ok":
                total_ok += 1
            elif status == "exists":
                total_skip += 1
            else:
                total_fail += 1

            with print_lock:
                print(
                    f"[{ts()}] "
                    f"[{job.year}-{job.month:02d} {job.month_idx}/{job.month_total}] "
                    f"{status:6s}  {job.name} -> {job.local_path}"
                )

    return (total_ok, total_skip, total_fail)


def main():
    print("Enter your FTP-enabled ESA credentials (the FTP account they enabled for you).")
    USER = input("FTP username (email): ").strip()
    PASS = getpass("FTP password: ")

    os.makedirs(DEST_ROOT, exist_ok=True)
    print(f"[INFO] Destination root: {DEST_ROOT}")
    print(f"[INFO] Parallel workers (simultaneous download workers): {MAX_WORKERS}")
    print(f"[INFO] Transfers: plain FTP forced = {FORCE_PLAIN_FTP}")

    t0 = time.time()
    grand_planned = grand_ok = grand_skip = grand_fail = 0

    for phase in PHASES:
        if not phase.get("enabled", True):
            continue

        jobs = build_jobs_for_phase(USER, PASS, phase)
        grand_planned += len(jobs)

        ok, skip, fail = run_jobs(USER, PASS, jobs)
        grand_ok += ok
        grand_skip += skip
        grand_fail += fail

        print(f"\n== PHASE COMPLETE: {phase['label']} ==")
        print(f"Planned:    {len(jobs)}")
        print(f"Downloaded: {ok}")
        print(f"Skipped:    {skip}")
        print(f"Failed:     {fail}")

    elapsed = int(time.time() - t0)
    print("\n== SUMMARY (ALL PHASES) ==")
    print(f"Planned:    {grand_planned}")
    print(f"Downloaded: {grand_ok}")
    print(f"Skipped:    {grand_skip}")
    print(f"Failed:     {grand_fail}")
    print(f"Elapsed:    {elapsed}s")
    print(f"Saved to:   {DEST_ROOT}")


if __name__ == "__main__":
    main()
